# ORACLE: Perturbation Analysis (RSP Module)

Use the Reversion Switch Predictor to identify minimal TF perturbation sets
that can switch cancer cells to the normal attractor basin.

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
import pandas as pd
import pickle
import anndata as ad
from pathlib import Path

DATA_DIR = Path('../data')
device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Device: {device}')

## 1. Load Data and Build CAM Output

In [ ]:
adata = ad.read_h5ad(DATA_DIR / 'processed/anndata/colorectal_GSE132465_processed.h5ad')
with open(DATA_DIR / 'processed/networks/colorectal_GSE132465_grn.pkl', 'rb') as f:
    grn = pickle.load(f)

# Load or recompute attractor results
from oracle.cam.preprocessing import CAMConfig
from oracle.cam.boolean_network import BooleanNetworkSimulator
from oracle.cam.continuous_ode import ContinuousGRNDynamics
from oracle.cam.attractor_classifier import AttractorClassifier

config = CAMConfig(cancer_type='colorectal', tissue='colon')
bool_net = BooleanNetworkSimulator(grn, config)
ode_model = ContinuousGRNDynamics(grn, config).to(device)

bool_attractors = bool_net.find_attractors(n_initial_states=2000)
genes = list(grn.nodes())
classifier = AttractorClassifier(cancer_type='colorectal', tissue='colon')
labels = classifier.classify(bool_attractors, genes)
cancer_attr, normal_attr = classifier.get_cancer_normal_pair(bool_attractors, labels)
basin_sizes = bool_net.compute_basin_sizes(bool_attractors, n_samples=10000)

print(f'Cancer attractor genes={len(cancer_attr) if cancer_attr is not None else 0}')
print(f'Normal attractor genes={len(normal_attr) if normal_attr is not None else 0}')

## 2. Train Cancer Score Function

In [ ]:
from oracle.rsp.cancer_score import CancerScoreFunction, RSPConfig

rsp_config = RSPConfig(n_genes=len(genes))
cancer_score_fn = CancerScoreFunction(rsp_config).to(device)

# Score attractors
for i, (attr, label) in enumerate(zip(bool_attractors, labels)):
    x = torch.tensor(attr.astype(np.float32)).unsqueeze(0).to(device)
    score = cancer_score_fn(x).item()
    print(f'Attractor {i} ({label}): cancer_score={score:.4f}')

## 3. GNN Switch Predictor

In [ ]:
from oracle.rsp.gnn_predictor import GNNSwitchPredictor

rsp_config = RSPConfig(n_genes=len(genes))
predictor = GNNSwitchPredictor(rsp_config)

if cancer_attr is not None:
    cancer_expr = {g: float(cancer_attr[i]) for i, g in enumerate(genes)}
    predictions = predictor.predict_switches(grn, cancer_expr)
    print(f'Top predicted switch genes:')
    for gene, score in sorted(predictions.items(), key=lambda x: x[1], reverse=True)[:10]:
        print(f'  {gene}: {score:.4f}')

## 4. Minimal Switch Optimization

In [ ]:
from oracle.rsp.switch_optimizer import MinimalSwitchOptimizer

optimizer = MinimalSwitchOptimizer(rsp_config)

if cancer_attr is not None and normal_attr is not None:
    switch_set = optimizer.optimize(
        cancer_attr, normal_attr, grn, ode_model, cancer_score_fn, genes
    )
    print(f'Minimal switch set ({len(switch_set.perturbations)} perturbations):')
    for gene, ptype in switch_set.perturbations.items():
        print(f'  {gene}: {ptype}')
    print(f'  Predicted reversion prob: {switch_set.predicted_reversion_probability:.4f}')
    print(f'  Delta cancer score: {switch_set.delta_cancer_score:.4f}')

## 5. Trajectory Simulation

In [ ]:
from oracle.rsp.perturbation_sim import PerturbationSimulator

sim = PerturbationSimulator(rsp_config)

if cancer_attr is not None and switch_set is not None:
    result = sim.simulate(
        initial_state=cancer_attr,
        perturbations=switch_set.perturbations,
        ode_model=ode_model,
        cancer_score_fn=cancer_score_fn,
        n_trajectories=20
    )
    print(f'Simulation results:')
    print(f'  Mean cancer score: {result.mean_cancer_score:.4f}')
    print(f'  Reversion fraction: {result.reversion_fraction:.4f}')
    print(f'  Delta score: {result.delta_score:.4f}')

    # Plot trajectories
    if result.trajectories is not None:
        plt.figure(figsize=(10, 4))
        for traj in result.trajectories[:5]:
            plt.plot(traj, alpha=0.7)
        plt.xlabel('Time step')
        plt.ylabel('Cancer score')
        plt.title('Cancer score trajectories under perturbation')
        plt.axhline(0.5, color='red', linestyle='--', label='Decision boundary')
        plt.legend()
        plt.show()

## 6. Perturbation Heatmap

In [ ]:
if switch_set is not None and cancer_attr is not None:
    perturbed_state = cancer_attr.copy()
    gene_idx = {g: i for i, g in enumerate(genes)}
    perturb_genes = list(switch_set.perturbations.keys())
    
    # Show expression changes at switch genes
    fig, ax = plt.subplots(figsize=(8, 4))
    data = []
    for gene in perturb_genes:
        idx = gene_idx.get(gene)
        if idx is not None:
            ptype = switch_set.perturbations[gene]
            before = float(cancer_attr[idx])
            after = 0.9 if 'Activation' in str(ptype) else 0.1
            data.append({'Gene': gene, 'Before': before, 'After': after,
                         'Perturbation': str(ptype)})
    
    df = pd.DataFrame(data)
    if not df.empty:
        x = range(len(df))
        ax.bar([i - 0.2 for i in x], df['Before'], width=0.4, label='Before', color='#F44336')
        ax.bar([i + 0.2 for i in x], df['After'], width=0.4, label='After', color='#2196F3')
        ax.set_xticks(list(x))
        ax.set_xticklabels(df['Gene'], rotation=45)
        ax.set_ylabel('Expression level')
        ax.set_title('Expression changes at switch genes')
        ax.legend()
        plt.tight_layout()
        plt.show()